In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
# 1. Define input/output paths
input_path = r"sentimentalanalysis\data\reduced\Subset 3.xlsx"
output_file = r"sentimentalanalysis\data\cleaned\Cleaned Subset 3.csv"

In [4]:
# 2. Load dataset
df = pd.read_excel(input_path, sheet_name="subset3")

# 3. Filter for Health Canada only
df = df[df["DEPT_E"] == "Health Canada"].copy()

# 4. Drop duplicate rows
df.drop_duplicates(inplace=True)

FileNotFoundError: [Errno 2] No such file or directory: 'sentimentalanalysis\\data\\reduced\\Subset 3.xlsx'

In [ ]:
# 5. Replace '9999' with NaN in sentiment columns
sentiment_cols = ["POSITIVE", "NEUTRAL", "NEGATIVE", "AGREE", "DISAGREE", "SCORE100"]
df[sentiment_cols] = df[sentiment_cols].replace(9999, np.nan)

# 6. Remove rows with ANSCOUNT = 0 or NaN
df = df[df["ANSCOUNT"].fillna(0) > 0]

# 7. Drop rows where all sentiment scores are missing
df = df.dropna(subset=sentiment_cols, how='all')

# 8. Keep only rows where sentiment adds up to ~100 (±5 tolerance)
df["SENTIMENT_TOTAL"] = df[["POSITIVE", "NEUTRAL", "NEGATIVE"]].sum(axis=1)
df = df[(df["SENTIMENT_TOTAL"] >= 95) & (df["SENTIMENT_TOTAL"] <= 105)]

# 9. Remove breakdown-only questions (e.g., Q117)
df = df[~df["QUESTION"].isin(["Q117", "Q118", "Q119"])]

# 10. Drop redundant bilingual or duplicate columns
df.drop(columns=[
    "descrip_F", "DEPT_F", "QUESTION (FR)", "QUESTIONTEXTFRENCH",
    "INDICATORFRA", "SUBINDICATORFRA"
], inplace=True, errors="ignore")

# 11. Clean text fields
text_fields = ["questiontext", "indicatoreng", "subindicatoreng"]
for col in text_fields:
    df[col] = df[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

# 12. Normalize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# 13. Convert organizational levels and demographics to category dtype
cat_fields = ["level1id", "level2id", "level3id", "level4id", "level5id", "demcode"]
df[cat_fields] = df[cat_fields].astype("category")

# 14. Remove rows with low response count (< 30)
df = df[df["anscount"] >= 30]

# 15. Reset index
df.reset_index(drop=True, inplace=True)

# 16. Save cleaned dataset
df.to_csv(output_file, index=False)

# 17. Preview
df.head()
